# 03 - Federated Client Distribution

Build a non-IID client partition over training consumers and quantify size and label heterogeneity. The same functions are used by the experiment suite.

In [ ]:
from pathlib import Path
import json
import sys

# Headless-safe plotting; Jupyter still renders figures inline.
import matplotlib
matplotlib.use("Agg")

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from preprocessing.pipeline import prepare_data
from federated.partition import build_partition, heterogeneity_report
from utilities.config import load_config

cfg = load_config("quick", [
    "data.synthetic.consumers=80",
    "data.synthetic.days=90",
    "data.synthetic.theft_rate=0.15",
    "clients.num_clients=5",
])
splits = prepare_data(cfg, seed=42, log=print)
partitions = build_partition(splits.train, cfg, seed=42)
rows = [p.summary(splits.train) for p in partitions]
display(pd.DataFrame(rows))  # noqa: F821
print(json.dumps(heterogeneity_report(partitions, splits.train), indent=2))


## Partition integrity

Every training window belongs to exactly one client.

In [ ]:
all_indices = np.concatenate([p.indices for p in partitions])
assert len(all_indices) == len(set(all_indices.tolist()))
assert len(all_indices) == len(splits.train)
assert set(map(str, np.concatenate([p.consumer_ids for p in partitions]))) == set(map(str, splits.train_consumers))
print("PASS: disjoint client windows and complete consumer coverage")


In [ ]:
frame = pd.DataFrame(rows)
ax = frame.set_index("client_id")[["n_samples", "n_theft_windows"]].plot(
    kind="bar", figsize=(8, 3), title="Client size and theft windows (SYNTHETIC DEMO DATA)")
ax.set_ylabel("windows")
plt.tight_layout()
plt.show()


The quick suite includes a client with zero theft windows. Adaptive aggregation gives that client neutral quality instead of collapsing its weight to the minimum; see `tests/test_aggregation.py`.